# SOPQuery — RAG Pipeline
## Sprint 1 · Stage 1: PDF Ingestion
### Functions
- Successfully load PDFs without error.
- Verify content of extracted text that is readible, no error present and non-empty.


In [1]:
# import libraries
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

/var/folders/my/5yhvxszd4rz8fvy2bm61s3_80000gn/T/ipykernel_32464/1752622643.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# load pdfs saved in data/FDA folder
# save all loaded documents into a variable: documents

pdf_path = Path("../data/FDA")

documents = []
for pdf in pdf_path.glob('*.pdf'):
    try:
        pdf_loader = PyPDFLoader(pdf)
        documents.extend(pdf_loader.load())
    
    # validate successful PDF ingestion
    except Exception as err:
        print(f'Failed to load {pdf.name}: {err}')

# check output
expected = len(list(pdf_path.glob('*.pdf')))
actual = len(set(doc.metadata['source'] for doc in documents))
print(f'Expected PDFs: {expected}')
print(f'Succefully loaded PDFs: {actual}')
print(documents[0].metadata)

Expected PDFs: 5
Succefully loaded PDFs: 5
{'producer': 'iTextSharp 4.0.3 (based on iText 2.0.2)', 'creator': 'CommonLook Office-2.1.15.47', 'creationdate': '2026-02-03T08:15:53-05:00', 'author': 'CDRH FDA', 'keywords': '', 'moddate': '2026-02-03T09:10:12-05:00', 'nccl_app': 'Office', 'nccl_standard': 'Section 508; WCAG 2.0 AA; PDF/UA', 'nccl_status': 'Passed', 'subject': '', 'title': 'Computer Software Assurance for Production and Quality Management System Software', 'part': '1', 'source': '../data/FDA/guidance-computer-software-assurance-production-quality-system.pdf', 'total_pages': 41, 'page': 0, 'page_label': '1'}


In [3]:
# print file name and number of pages of each files
file_info = set((doc.metadata['title'], doc.metadata['total_pages']) for doc in documents)

for title, pages in file_info:
    print(f'{title}: {pages} pages')

Computer Software Assurance for Production and Quality Management System Software: 41 pages
SOPP 8201: Administrative Processing of Clinical Holds for Investigational New Drug Applications: 15 pages
SOPP 8212: Breakthrough Therapy Products - Designation and Management: 25 pages
SOPP 8217: Administrative Process and Review Management of Investigational New Drug Applications: 31 pages
Investigating Out-of-Specification (OOS) Test Results for Pharmaceutical Production: 17 pages


In [4]:
# spot-check first 500 characters in first document
print(documents[0].page_content[:500])

Contains Nonbinding Recommendations
Computer Software Assurance for 
Production and Quality Management 
System Software
Guidance for Industry and 
Food and Drug Administration Staff 
Document issued on February 3, 2026.
This document supersedes “Computer Software Assurance for Production 
and Quality System Software,” issued September 24, 2025. 
For questions about this document regarding CDRH-regulated devices, contact the Compliance 
and Quality Staff at 301-796-5577 or by email at CaseforQual


## Sprint 1 · Stage 2: Chunking
### Functions
- Split documents into chunks that can be quickly retrieved and integrated into the model prompt.
- Configure chunk_size and overlap parameters
- Inspect chunk content that each chunk size is within the chunk_size character limit

In [5]:
# import library
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

# print the first 3 chunks
for i, chunk in enumerate(chunks[:3]):
    print(f'\n------ Chunk {i+1} ------')

    # Chunk size remains within 500 characters limit
    print(f'Characters: {len(chunk.page_content)}')
    print(f'Source: {chunk.metadata['source']}')
    print(f'Page: {chunk.metadata['page']}')
    # inspect chunk output
    print(chunk.page_content[:500])


------ Chunk 1 ------
Characters: 438
Source: ../data/FDA/guidance-computer-software-assurance-production-quality-system.pdf
Page: 0
Contains Nonbinding Recommendations
Computer Software Assurance for 
Production and Quality Management 
System Software
Guidance for Industry and 
Food and Drug Administration Staff 
Document issued on February 3, 2026.
This document supersedes “Computer Software Assurance for Production 
and Quality System Software,” issued September 24, 2025. 
For questions about this document regarding CDRH-regulated devices, contact the Compliance

------ Chunk 2 ------
Characters: 459
Source: ../data/FDA/guidance-computer-software-assurance-production-quality-system.pdf
Page: 0
and Quality Staff at 301-796-5577 or by email at CaseforQuality@fda.hhs.gov. For questions 
about this document regarding CBER-regulated devices, contact the Office of Communication, 
Outreach, and Development (OCOD) at 800-835-4709 or 240-402-8010, or by email at 
industry.biologics@fda.hhs.